# Phase 0: Health Checks

This notebook provides diagnostic visualizations for bidless simulation datasets.
It helps verify data integrity, detect biases, and explore feature-label relationships.

**Sections:**
0. Health Scorecard (quick pass/warn/fail summary)
1. Run Summary & Data Loading
2. Dataset Integrity Checks
3. Strata Completeness
4. By Contract/Trump Analysis
5. By Seat Analysis
6. Feature Distributions
7. Feature-Label Relationships
8. Time/Batch Drift Analysis

## Configuration

Set the path to your dataset directory here:

In [ ]:
# === CONFIGURATION ===

# --- Data Source Mode ---
DEMO_MODE = False  # If True, generates synthetic data; if False, loads from RUN_DIR

# If DEMO_MODE=False, set this path:
RUN_DIR = "../../data/runs/YOUR_RUN_ID"  # Will load from RUN_DIR/datasets/

# --- Demo Mode Parameters (used when DEMO_MODE=True) ---
DEMO_SEED = 42
DEMO_N_DEALS = 2000  # Minimum for bias detection per rigor standards

# --- Analysis Parameters ---
ROLLING_WINDOW = 100  # Window size for rolling mean plots
TOP_FEATURES = 9      # Number of features to show in distribution grid

## Experimental Setup: Phase 0 Bidless

**What is Phase 0?**
- **No bidding phase**: Contracts and trumps are assigned exogenously (not chosen through bidding)
- **Scenario-driven assignment**: Contracts/trumps come from explicit scenario list in experiment config (not uniform random sampling)
- **Policy-dependent outcomes**: The `tricks_won` values depend on the play strategy used (e.g., RandomLegalStrategy, GreedyStrategy)
- **Determinism**: Seed controls both deal generation and any strategy randomness

**This dataset:**
- Strategy: Check metadata for `strategies` field (typically RandomLegalStrategy with seed)
- Scenarios: Check metadata for `scenarios` field (typically 6: suit×4 trumps + high + low)
- Deals per scenario: Check metadata for `n_per` field

See `meta.json` or use `load_meta()` to inspect these values.

In [ ]:
# Standard imports
import sys
from pathlib import Path

import pandas as pd

# Add src to path for local development
project_root = Path.cwd().parent.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Diagnostic utilities
import matplotlib.pyplot as plt

from bid_euchre.diagnostics import (
    compare_first_last_batch,
    compute_health_scorecard,
    compute_seat_balance,
    display_scorecard,
    load_bidless_dataset,
    load_meta,
    plot_feature_correlation,
    plot_feature_distributions,
    plot_hand_value_by_contract,
    plot_hand_value_by_seat,
    plot_rolling_mean,
)
from bid_euchre.diagnostics.charts import plot_feature_vs_label
from bid_euchre.diagnostics.loaders import get_dataset_summary
from bid_euchre.diagnostics.stats import (
    compute_correlation_with_label,
    compute_feature_stats,
)

# Optional: seaborn for enhanced visualizations
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    print("seaborn not available, using matplotlib defaults")

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Imports successful!")

In [ ]:
# ============================================================================
# DATA GENERATION FACTORY (for demo mode)
# ============================================================================

def build_demo_dataset(seed: int, n_deals: int) -> pd.DataFrame:
    """Build demo bidless dataset with features only (no simulation).

    Generates data for all 6 Phase 0 scenarios: 4 suit contracts + high + low.

    Args:
        seed: Random seed for reproducibility
        n_deals: Number of deals per scenario

    Returns:
        DataFrame with hand_id, seat, contract_type, trump_suit, feat_* columns
    """
    from bid_euchre.features.hand_eval import get_hand_features
    from bid_euchre.sim.deals import generate_deal

    contract_types = ['suit', 'suit', 'suit', 'suit', 'high', 'low']
    trumps = ['C', 'D', 'H', 'S', None, None]  # Aligned with contract_types

    hands_data = []
    for deal_id in range(n_deals):
        hands = generate_deal(seed, deal_id)

        for contract_type, trump in zip(contract_types, trumps):
            for seat in range(4):
                hand = hands[seat]
                features = get_hand_features(hand, contract_type, trump)
                hands_data.append({
                    'hand_id': f"{deal_id}_{contract_type}_{trump}",
                    'seat': seat,
                    'contract_type': contract_type,
                    'trump_suit': trump if contract_type == 'suit' else None,
                    **{f'feat_{k}': v for k, v in features.items()}
                })

    return pd.DataFrame(hands_data)


print("✅ Demo data factory loaded")

**Data Source Options:**

1. **Production mode** (`DEMO_MODE=False`):
   - Point `RUN_DIR` to an existing experiment run
   - Generate production dataset with:
     ```bash
     PYTHONPATH=src python experiments/run_experiment.py \
       --config experiments/configs/bidless_dataset_collection.yaml \
       --seed 42 \
       --n_per 2000
     ```
   - Then set `RUN_DIR = "../../data/runs/<your_run_id>"`

2. **Demo mode** (`DEMO_MODE=True`):
   - Generates synthetic data in-memory
   - Uses `DEMO_N_DEALS=2000` (meets rigor standards for bias detection)
   - Useful for testing, development, or when no pre-existing dataset available

In [ ]:
# ============================================================================
# Data Loading / Generation
# ============================================================================

if not DEMO_MODE:
    print("📂 Loading existing dataset from RUN_DIR...")
    from pathlib import Path

    dataset_path = Path(RUN_DIR) / "datasets"
    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {dataset_path}\n"
            f"Set RUN_DIR to your run directory, or use DEMO_MODE=True"
        )

    df = load_bidless_dataset(dataset_path)
    print(f"✅ Loaded {len(df):,} rows from {dataset_path}")

else:
    print("🎭 DEMO_MODE=True: Generating synthetic data...")
    print(f"   Seed: {DEMO_SEED}")
    print(f"   Deals per scenario: {DEMO_N_DEALS}")
    print("   Scenarios: 6 (4 suit + high + low)")

    df = build_demo_dataset(seed=DEMO_SEED, n_deals=DEMO_N_DEALS)
    dataset_path = None  # No disk path for demo data

    print(f"\n✅ Generated {len(df):,} rows (synthetic demo data)")
    print(f"   Expected: {DEMO_N_DEALS * 6 * 4:,} rows (n_deals × scenarios × seats)")

---
## Section 0: Health Scorecard

Quick pass/warn/fail summary of dataset health.

In [ ]:
# Compute and display health scorecard
scorecard = compute_health_scorecard(df)
print(display_scorecard(scorecard))

---
## Section 1: Run Summary & Data Loading

Overview of the loaded dataset.

In [ ]:
# Load metadata if available
if dataset_path is not None:
    try:
        meta = load_meta(dataset_path)
        print("=== Metadata ===")
        for key, value in meta.items():
            print(f"  {key}: {value}")
    except FileNotFoundError:
        print("No metadata file found (bidless_meta.json)")
else:
    print("=== Metadata ===")
    print("  No metadata (demo mode)")

# Dataset summary
print("\n=== Dataset Summary ===")
summary = get_dataset_summary(df)
for key, value in summary.items():
    if key != 'feature_columns':
        print(f"  {key}: {value}")

print(f"\n  Feature columns ({len(summary['feature_columns'])}):")  
for col in summary['feature_columns'][:10]:
    print(f"    - {col}")
if len(summary['feature_columns']) > 10:
    print(f"    ... and {len(summary['feature_columns']) - 10} more")

In [ ]:
# Preview the data
print("=== Data Preview ===")
display(df.head(8))

---
## Section 2: Dataset Integrity Checks

Detailed integrity verification.

In [ ]:
# Check (hand_id, seat) uniqueness
print("=== Row Uniqueness ===")
duplicates = df.duplicated(subset=['hand_id', 'seat']).sum()
print(f"  Duplicate (hand_id, seat) pairs: {duplicates}")
status = '\u2705 PASS' if duplicates == 0 else '\u274c FAIL'
print(f"  Status: {status}")

# Check seats per hand
print("\n=== Seats Per Hand ===")
seats_per_hand = df.groupby('hand_id').size()
print(f"  Hands with exactly 4 seats: {(seats_per_hand == 4).sum()}")
print(f"  Hands with != 4 seats: {(seats_per_hand != 4).sum()}")

# Check for NaN values in features
print("\n=== NaN Values in Features ===")
feat_cols = [c for c in df.columns if c.startswith('feat_')]
nan_counts = df[feat_cols].isna().sum()
total_nans = nan_counts.sum()
print(f"  Total NaN values: {total_nans}")
if total_nans > 0:
    print("  Columns with NaN:")
    for col, count in nan_counts[nan_counts > 0].items():
        print(f"    {col}: {count}")

---
## Section 3: Strata Completeness

Check that all contract types, trump suits, and seats are balanced.

In [ ]:
print("=== Strata Completeness (Contract × Trump × Seat) ===")
print("\nCounts by contract type:")
contract_counts = df.groupby('contract_type').size()
display(contract_counts)

print("\nCounts by trump suit (suit contracts only):")
suit_df = df[df['contract_type'] == 'suit']
if len(suit_df) > 0:
    trump_counts = suit_df.groupby('trump_suit').size()
    display(trump_counts)
    
    # Check balance
    expected_per_trump = len(suit_df) / suit_df['trump_suit'].nunique()
    imbalance = trump_counts.max() - trump_counts.min()
    if imbalance > expected_per_trump * 0.1:  # 10% tolerance
        print(f"⚠️  Imbalance detected: {imbalance} row difference across trumps")
    else:
        print("✅ Trump distribution looks balanced")

print("\nCounts by seat:")
seat_counts = df.groupby('seat').size()
display(seat_counts)

# Check seat balance
expected_per_seat = len(df) / df['seat'].nunique()
imbalance = seat_counts.max() - seat_counts.min()
if imbalance > expected_per_seat * 0.05:  # 5% tolerance
    print(f"⚠️  Seat imbalance: {imbalance} row difference")
else:
    print("✅ Seat counts balanced")

---
## Section 4: By Contract/Trump Analysis

Hand value distributions by contract type and trump suit.

In [ ]:
# Hand value by contract type
fig = plot_hand_value_by_contract(df)
plt.show()

# Statistics by contract
print("\n=== Statistics by Contract Type ===")
contract_stats = df.groupby('contract_type')['feat_hand_value'].agg(['count', 'mean', 'std', 'min', 'max'])
display(contract_stats)

In [ ]:
# Trump suit analysis (for suit contracts only)
suit_df = df[df['contract_type'] == 'suit']
if len(suit_df) > 0:
    print("=== Hand Value by Trump Suit (Suit Contracts Only) ===")
    trump_stats = suit_df.groupby('trump_suit')['feat_hand_value'].agg(['count', 'mean', 'std'])
    display(trump_stats)
    
    # Plot if seaborn available
    fig, ax = plt.subplots(figsize=(10, 5))
    try:
        import seaborn as sns
        sns.boxplot(data=suit_df, x='trump_suit', y='feat_hand_value', ax=ax)
    except ImportError:
        suit_df.boxplot(column='feat_hand_value', by='trump_suit', ax=ax)
    ax.set_title('Hand Value by Trump Suit')
    ax.set_xlabel('Trump Suit')
    ax.set_ylabel('Hand Value')
    plt.tight_layout()
    plt.show()
else:
    print("No suit contracts in dataset")

---
## Section 5: By Seat Analysis

**Critical check:** If seats 1-3 look identical to seat 0, the per-seat feature bug may exist!

In [ ]:
# Hand value by seat
fig = plot_hand_value_by_seat(df)
plt.show()

# Seat balance check
balance = compute_seat_balance(df)
print("\n=== Seat Balance ===")
print(f"  Global mean: {balance.global_mean:.4f}")
print("  Seat means:")
for seat, mean in sorted(balance.seat_means.items()):
    dev = abs(mean - balance.global_mean)
    print(f"    Seat {seat}: {mean:.4f} (deviation: {dev:.4f})")
print(f"  Max deviation: {balance.max_deviation:.4f} (seat {balance.max_deviation_seat})")
balanced_status = '\u2705 Yes' if balance.is_balanced else '\u26a0\ufe0f No'
print(f"  Balanced: {balanced_status}")

In [ ]:
# Team analysis (seats 0,2 vs 1,3)
df['team'] = df['seat'].apply(lambda s: 0 if s in [0, 2] else 1)

print("=== Team Comparison ===")
team_stats = df.groupby('team')['feat_hand_value'].agg(['count', 'mean', 'std'])
display(team_stats)

fig, ax = plt.subplots(figsize=(8, 5))
try:
    import seaborn as sns
    sns.boxplot(data=df, x='team', y='feat_hand_value', ax=ax)
except ImportError:
    df.boxplot(column='feat_hand_value', by='team', ax=ax)
ax.set_xticklabels(['Team 0 (seats 0,2)', 'Team 1 (seats 1,3)'])
ax.set_title('Hand Value by Team')
ax.set_xlabel('')
ax.set_ylabel('Hand Value')
plt.tight_layout()
plt.show()

---
## Section 6: Feature Distributions

Histograms of key features.

In [ ]:
# Feature distribution grid
fig = plot_feature_distributions(df, figsize=(14, 12))
plt.show()

In [ ]:
# Feature statistics table
print("=== Feature Statistics ===")
stats_df = compute_feature_stats(df)
display(stats_df)

---
## Section 7: Feature-Label Relationships

Correlation analysis and scatter plots.

In [ ]:
# Correlation heatmap
fig = plot_feature_correlation(df)
plt.show()

In [ ]:
# Correlation with hand_value
print("=== Feature Correlations with hand_value ===")
corr_df = compute_correlation_with_label(df, 'feat_hand_value')
display(corr_df.head(15))

In [ ]:
# Trump count vs hand value (should show strong positive correlation for suit contracts)
if 'feat_trump_count' in df.columns:
    fig = plot_feature_vs_label(df, 'trump_count', 'hand_value')
    plt.show()

---
## Section 8: Time/Batch Drift Analysis

Check for drift over the course of data collection.

In [ ]:
# Rolling mean of hand_value
fig = plot_rolling_mean(df, 'feat_hand_value', window=ROLLING_WINDOW)
plt.show()

In [ ]:
# First vs last batch comparison
comparison = compare_first_last_batch(df, 'feat_hand_value', batch_fraction=0.1)

print("=== First vs Last Batch (10% each) ===")
print(f"  First batch mean: {comparison.first_mean:.4f}")
print(f"  Last batch mean: {comparison.last_mean:.4f}")
print(f"  Difference: {comparison.difference:.4f} ({comparison.percent_change:+.1f}%)")
if comparison.mannwhitney_pvalue is not None:
    print(f"  Mann-Whitney U p-value: {comparison.mannwhitney_pvalue:.4f}")
    sig_status = 'Yes' if comparison.is_significant else 'No'
    print(f"  Statistically significant: {sig_status}")
print(f"\n  Interpretation: {comparison.interpretation}")

In [ ]:
# Visualize first vs last batch
n = len(df)
batch_size = int(n * 0.1)

df['batch'] = 'Middle'
df.iloc[:batch_size, df.columns.get_loc('batch')] = 'First 10%'
df.iloc[-batch_size:, df.columns.get_loc('batch')] = 'Last 10%'

fig, ax = plt.subplots(figsize=(10, 5))
try:
    import seaborn as sns
    sns.boxplot(data=df[df['batch'] != 'Middle'], x='batch', y='feat_hand_value', 
                order=['First 10%', 'Last 10%'], ax=ax)
except ImportError:
    batch_df = df[df['batch'] != 'Middle']
    batch_df.boxplot(column='feat_hand_value', by='batch', ax=ax)
ax.set_title('Hand Value: First vs Last Batch')
ax.set_xlabel('')
ax.set_ylabel('Hand Value')
plt.tight_layout()
plt.show()

# Clean up temporary column
df.drop('batch', axis=1, inplace=True)

---
## Summary

Final health status and key findings.

In [ ]:
print("=" * 60)
print("FINAL HEALTH SUMMARY")
print("=" * 60)

summary = scorecard.summary()
if summary['FAIL'] == 0 and summary['WARN'] == 0:
    print("\n\u2705 ALL CHECKS PASSED - Dataset looks healthy!")
elif summary['FAIL'] == 0:
    print(f"\n\u26a0\ufe0f  {summary['WARN']} WARNING(S) - Review warnings above")
else:
    print(f"\n\u274c {summary['FAIL']} FAILURE(S) - Fix issues before proceeding")

print(f"\n  Passed: {summary['PASS']}")
print(f"  Warnings: {summary['WARN']}")
print(f"  Failures: {summary['FAIL']}")

print("\n" + "=" * 60)